# MWS Vision Retrieval Filter

Notebook for turning MTSAIR/MWS-Vision-Bench into question-image retrieval pairs.

## 0. Installations

In [1]:
%pip install -q -U datasets "pillow<12" "pandas<3" "torch==2.10.0" tqdm accelerate transformers qwen-vl-utils

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 45.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


## 1. Setup

In [30]:
from pathlib import Path
from io import BytesIO
import hashlib, json, random, re, time

import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from datasets import load_dataset

DATASET_ID = "MTSAIR/MWS-Vision-Bench"
OUT = Path("outputs/mws_vision_retrieval")
IMG_DIR = OUT / "images"
AUDIT = OUT / "filter_audit.jsonl"
PAIRS = OUT / "pairs.jsonl"
HUMAN_LABELS_PATH = OUT / "human_label_template.csv"
METRICS = OUT / "filter_metrics.json"

MAX_SAMPLES = None # None for all rows
PROMPT_TEST_SAMPLE_SIZE = 50
RANDOM_SEED = 42

VLM_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
RUN_PROMPT_TEST = True
RUN_FULL_FILTER = True   # flip after prompt test is ok

OUT.mkdir(parents=True, exist_ok=True)
IMG_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load dataset

In [10]:
def save_image(img):
    img = img.convert("RGB")
    buf = BytesIO()
    img.save(buf, format="PNG")
    image_id = "img_" + hashlib.sha1(buf.getvalue()).hexdigest()[:16]
    path = IMG_DIR / f"{image_id}.png"
    if not path.exists():
        img.save(path)
    return image_id, str(path), img.size


ds = load_dataset(DATASET_ID, split="train")
idxs = range(len(ds)) if MAX_SAMPLES is None else range(min(MAX_SAMPLES, len(ds)))
rows = []

for i in tqdm(idxs):
    ex = ds[i]
    image_id, image_path, (w, h) = save_image(ex["image"])
    rows.append({
        "query_id": str(ex.get("id", i)),
        "image_id": image_id,
        "question": ex["question"],
        "image_path": image_path,
        "source_type": ex.get("type", ""),
        "dataset_name": ex.get("dataset_name", ""),
        "answers": ex.get("answers", []),
        "image_width": w,
        "image_height": h,
    })

candidates = pd.DataFrame(rows)
display(candidates.head())
display(candidates.groupby("source_type").size().reset_index(name="count"))

  0%|          | 0/1302 [00:00<?, ?it/s]

,query_id,image_id,question,image_path,source_type,dataset_name,answers,image_width,image_height
0,1,img_8d1725a1bd972fcc,Где находится герб на документе? Выведи абсолю...,outputs/mws_vision_retrieval/images/img_8d1725...,text grounding ru,business,"[398, 65, 467, 140]",905,1280
1,2,img_8d1725a1bd972fcc,Какое качество воды нужно обеспечить согласно ...,outputs/mws_vision_retrieval/images/img_8d1725...,reasoning VQA ru,business,[соответствующее гигиеническим нормативам],905,1280
2,3,img_8d1725a1bd972fcc,Экспортируй текст с изображения и представь ег...,outputs/mws_vision_retrieval/images/img_8d1725...,full-page OCR ru,business,[ФЕДЕРАЛЬНАЯ СЛУЖБА ПО НАДЗОРУ В СФЕРЕ ЗАЩИТЫ ...,905,1280
3,4,img_8d1725a1bd972fcc,О чем говорится в данном документе. Дай оттекс...,outputs/mws_vision_retrieval/images/img_8d1725...,document parsing ru,business,[<center>ФЕДЕРАЛЬНАЯ СЛУЖБА ПО НАДЗОРУ В СФЕРЕ...,905,1280
4,5,img_45351385156849f6,"На документе найди ячейку таблицы, содержащую ...",outputs/mws_vision_retrieval/images/img_453513...,text grounding ru,business,"[615, 1052, 726, 1098]",892,1280


,source_type,count
0,document parsing ru,243
1,full-page OCR ru,144
2,key information extraction ru,119
3,reasoning VQA ru,400
4,text grounding ru,396


## 3. VLM filter

In [37]:
PROMPT = """
Ты оцениваешь изображение документа и вопрос.
Нужно понять, годится ли вопрос для retrieval: правильным результатом должно быть именно это изображение.

true: без именно этого документа нельзя уверенно ответить.
false: вопрос общий/шаблонный и подходит почти к любому похожему документу.

false-примеры: "Распознай весь текст", "Передай в markdown", "О чем документ".
true-примеры: "Какой размер оплаты?", "Найди ячейку с 5 495,00", "Как называется раздел 8?".

Вопрос: {question}

Ответь только JSON:
{{"is_unique_to_image": true или false, "reason": "кратко"}}
""".strip()


def read_jsonl(path):
    if not path.exists():
        return []
    return [json.loads(x) for x in path.read_text(encoding="utf-8").splitlines() if x.strip()]


def append_jsonl(path, row):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def cache():
    return {str(x["query_id"]): x for x in read_jsonl(AUDIT)}


def load_vlm():
    from transformers import AutoProcessor
    from transformers import Qwen3VLForConditionalGeneration as Model
    processor = AutoProcessor.from_pretrained(VLM_MODEL_ID, trust_remote_code=True)
    model = Model.from_pretrained(VLM_MODEL_ID, device_map="auto", torch_dtype="auto", trust_remote_code=True).eval()
    return processor, model


def parse_answer(text):
    fence = chr(96) * 3 #=```
    text = text.strip().replace(fence + "json", "").replace(fence, "").strip() #=``` или ```json
    text = re.search(r"\{.*\}", text, re.S).group(0)
    obj = json.loads(text)
    return bool(obj["is_unique_to_image"]), str(obj.get("reason", ""))


def judge(row, processor, model):
    msg = [{"role": "user", "content": [
        {"type": "image", "image": str(Path(row.image_path).resolve())},
        {"type": "text", "text": PROMPT.format(question=row.question)},
    ]}]
    inputs = processor.apply_chat_template(msg, tokenize=True, return_dict=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=100)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
    raw = processor.batch_decode(trimmed, skip_special_tokens=True)[0]
    ok, reason = parse_answer(raw)
    return {
        "query_id": row.query_id,
        "image_id": row.image_id,
        "question": row.question,
        "source_type": row.source_type,
        "dataset_name": row.dataset_name,
        "vlm_is_unique": ok,
        "vlm_reason": reason,
        "filter_model": VLM_MODEL_ID,
        "parse_ok": True,
        "raw_output": raw
    }


def run_filter(df):
    c = cache()
    valid_c = {k: v for k, v in c.items() if v.get("parse_ok", True)}
    missing = df[~df.query_id.isin(valid_c.keys())]
    print("cached (valid)", len(valid_c), "missing", len(missing))
    if len(missing):
        processor, model = load_vlm()
        for row in tqdm(missing.itertuples(index=False), total=len(missing)):
            try:
                decision = judge(row, processor, model)
            except Exception as e:
                decision = {"query_id": row.query_id, "image_id": row.image_id, "question": row.question, "source_type": row.source_type, "dataset_name": row.dataset_name, "vlm_is_unique": False, "vlm_reason": "parse_failed", "filter_model": VLM_MODEL_ID, "parse_ok": False, "error": repr(e)}
            append_jsonl(AUDIT, decision)
            c[decision["query_id"]] = decision
    return c

## 4. Small sample to test prompt

In [38]:
sample = candidates.sample(min(PROMPT_TEST_SAMPLE_SIZE, len(candidates)), random_state=RANDOM_SEED)

decisions = run_filter(sample) if RUN_PROMPT_TEST else cache()

label_rows = []
for row in sample.itertuples(index=False):
    d = decisions.get(row.query_id, {})
    label_rows.append({
        "query_id": row.query_id,
        "image_id": row.image_id,
        "question": row.question,
        "image_path": row.image_path,
        "source_type": row.source_type,
        "dataset_name": row.dataset_name,
        "vlm_is_unique": d.get("vlm_is_unique"),
        "vlm_reason": d.get("vlm_reason", ""),
        "human_is_unique": "",
        "human_notes": "",
    })

label_df = pd.DataFrame(label_rows)
label_df.to_csv(HUMAN_LABELS_PATH, index=False)
print(HUMAN_LABELS_PATH.resolve())
display(label_df.head())

cached (valid) 53 missing 17


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

  0%|          | 0/17 [00:00<?, ?it/s]

/content/outputs/mws_vision_retrieval/human_label_template.csv


,query_id,image_id,question,image_path,source_type,dataset_name,vlm_is_unique,vlm_reason,human_is_unique,human_notes
0,487,img_120048f4a11a18b0,"Найди значения для ключей:\n""2001 г."",\n""2002 ...",outputs/mws_vision_retrieval/images/img_120048...,key information extraction ru,business,True,Значения для указанных годов присутствуют толь...,,
1,733,img_3c7747a08a848b40,Выведи изображение в формате markdown,outputs/mws_vision_retrieval/images/img_3c7747...,document parsing ru,business,False,Вопрос общий и шаблонный — 'выведи в формате m...,,
2,321,img_ad05fe7079825604,Прочти текст на этом изображении и передай его...,outputs/mws_vision_retrieval/images/img_ad05fe...,full-page OCR ru,business,False,"Вопрос общий и шаблонный, подходит к любому до...",,
3,860,img_5817674aaaa96643,"Назови любое из действий, которое указано в пу...",outputs/mws_vision_retrieval/images/img_581767...,reasoning VQA ru,business,True,Вопрос требует привести формулировку из конкре...,,
4,1293,img_e236761638c9a034,Какой пол пациента указан в документе? В ответ...,outputs/mws_vision_retrieval/images/img_e23676...,reasoning VQA ru,personal,True,Вопрос требует указания конкретного поля из до...,,


## 5. Prompt evaluation

In [41]:
def to01(x):
    if pd.isna(x):
        return None
    if str(x).strip().lower() == "true":
        return 1
    if str(x).strip().lower() == "false":
        return 0
    return None


def calc(y, p):
    tp = sum((y == 1) & (p == 1)); tn = sum((y == 0) & (p == 0))
    fp = sum((y == 0) & (p == 1)); fn = sum((y == 1) & (p == 0))
    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    accuracy = (tp + tn) / len(y) if len(y) else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    return {"n": int(len(y)), "precision": precision, "recall": recall, "accuracy": accuracy, "f1": f1, "confusion_matrix": {"tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)}}

labels = pd.read_csv(HUMAN_LABELS_PATH)
labels["human"] = labels.human_is_unique.map(to01)
labels["pred"] = labels.vlm_is_unique.map(to01)
labels = labels.dropna(subset=["human", "pred"])

if len(labels) == 0:
    print("No human labels yet. Fill human_is_unique and rerun.")
else:
    labels["human"] = labels.human.astype(int)
    labels["pred"] = labels.pred.astype(int)
    metrics = {
        "overall": calc(labels.human, labels.pred),
        "per_source_type": {k: calc(g.human, g.pred) for k, g in labels.groupby("source_type")},
    }
    METRICS.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(metrics["overall"], ensure_ascii=False, indent=2))
    display(pd.DataFrame([{k: v for k, v in metrics["overall"].items() if k != "confusion_matrix"}]))

{
  "n": 50,
  "precision": 1.0,
  "recall": 0.6451612903225806,
  "accuracy": 0.78,
  "f1": 0.7843137254901961,
  "confusion_matrix": {
    "tp": 20,
    "tn": 19,
    "fp": 0,
    "fn": 11
  }
}


,n,precision,recall,accuracy,f1
0,50,1.0,0.645161,0.78,0.784314


## 6. Full filtering to make dataset

In [42]:
if RUN_FULL_FILTER:
    decisions = run_filter(candidates)
else:
    decisions = cache()
    print("using cached decisions only; set RUN_FULL_FILTER=True for full run")

pairs = []
summary = []
for row in candidates.itertuples(index=False):
    d = decisions.get(row.query_id)
    status = "missing"
    if d and d.get("parse_ok") and d.get("vlm_is_unique") is True:
        status = "accepted"
        pairs.append({
            "query_id": row.query_id,
            "image_id": row.image_id,
            "query": row.question,
            "positive_image_path": row.image_path,
            "source_type": row.source_type,
            "dataset_name": row.dataset_name,
            "answers": row.answers,
            "filter_model": d.get("filter_model", VLM_MODEL_ID),
            "filter_reason": d.get("vlm_reason", ""),
        })
    elif d:
        status = "rejected" if d.get("parse_ok") else "parse_failed"
    summary.append({"query_id": row.query_id, "source_type": row.source_type, "status": status})

with PAIRS.open("w", encoding="utf-8") as f:
    for x in pairs:
        f.write(json.dumps(x, ensure_ascii=False) + "\n")

print("pairs:", len(pairs), "->", PAIRS.resolve())
display(pd.DataFrame(summary).groupby(["source_type", "status"]).size().reset_index(name="count"))
if pairs:
    display(pd.DataFrame(pairs).head())

cached (valid) 70 missing 1232


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

  0%|          | 0/1232 [00:00<?, ?it/s]

pairs: 714 -> /content/outputs/mws_vision_retrieval/pairs.jsonl


,source_type,status,count
0,document parsing ru,accepted,3
1,document parsing ru,rejected,240
2,full-page OCR ru,accepted,13
3,full-page OCR ru,rejected,131
4,key information extraction ru,accepted,103
5,key information extraction ru,rejected,16
6,reasoning VQA ru,accepted,332
7,reasoning VQA ru,parse_failed,7
8,reasoning VQA ru,rejected,61
9,text grounding ru,accepted,263


,query_id,image_id,query,positive_image_path,source_type,dataset_name,answers,filter_model,filter_reason
0,2,img_8d1725a1bd972fcc,Какое качество воды нужно обеспечить согласно ...,outputs/mws_vision_retrieval/images/img_8d1725...,reasoning VQA ru,business,[соответствующее гигиеническим нормативам],Qwen/Qwen3-VL-8B-Instruct,Вопрос ссылается на конкретный пункт (1.1) и т...
1,5,img_45351385156849f6,"На документе найди ячейку таблицы, содержащую ...",outputs/mws_vision_retrieval/images/img_453513...,text grounding ru,business,"[615, 1052, 726, 1098]",Qwen/Qwen3-VL-8B-Instruct,"Число '5 495,00' встречается только в одной яч..."
2,6,img_45351385156849f6,"На основе представленной таблицы, необходимо н...",outputs/mws_vision_retrieval/images/img_453513...,key information extraction ru,business,"[{'Текущий ремонт кровли.': ['85,00'], 'Гермет...",Qwen/Qwen3-VL-8B-Instruct,Запрашиваемые ключи и их значения в столбце 'П...
3,7,img_45351385156849f6,Затраты на какой вид работ составили 109 тыс. ...,outputs/mws_vision_retrieval/images/img_453513...,reasoning VQA ru,business,[текущий ремонт кровли],Qwen/Qwen3-VL-8B-Instruct,"В таблице документа в строке 7 указано, что за..."
4,11,img_ec60d7c9d2003c3b,За сколько часов до назначенного времени перев...,outputs/mws_vision_retrieval/images/img_ec60d7...,reasoning VQA ru,business,[три],Qwen/Qwen3-VL-8B-Instruct,В документе в пункте 2.8 указано конкретное тр...


## 7. Result

In [45]:
pairs_df = pd.read_json(PAIRS, lines=True) if PAIRS.exists() and PAIRS.stat().st_size else pd.DataFrame()
print("pairs", len(pairs_df))
if len(pairs_df):
    print("unique images", pairs_df.image_id.nunique())
    display(pairs_df.groupby("source_type").size().reset_index(name="accepted_count"))

pairs 714
unique images 373


,source_type,accepted_count
0,document parsing ru,3
1,full-page OCR ru,13
2,key information extraction ru,103
3,reasoning VQA ru,332
4,text grounding ru,263
